[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/08-rural-community-mapping.ipynb)

# Rural Community Mapping: Defining Communities by Drive Time

## Three Hub Towns Across Western Kansas

County lines are administrative accidents. They were drawn by territorial legislators in the 19th century to organize land records, not to describe how people actually live. In rural Kansas, a county seat of 800 people may share a county with farmland stretching 30 miles in every direction, while the nearest hospital, high school, or grocery store sits in a hub town across the county line.

This notebook uses **drive-time isochrones** to define functional rural communities -- the areas whose residents realistically depend on a given town for services. We compare three western Kansas hub towns:

- **Hays** (~21,000) -- a university town (Fort Hays State) and regional medical center
- **Dodge City** (~28,000) -- an agricultural and meatpacking center on the Arkansas River
- **Liberal** (~20,000) -- a meatpacking and energy town near the Oklahoma border

These towns are spaced 100--150 miles apart along the Great Plains corridor. Each draws workers, patients, and shoppers from multiple surrounding counties. By generating **25-minute driving isochrones** (longer than the 15-minute urban standard, reflecting rural reality), we can see how far each town's functional community actually extends -- and compare that to the arbitrary county boundaries.

By the end of this notebook, you will know how to:

- Generate driving isochrones to define rural service areas
- Compare isochrone coverage against county-scale benchmarks
- Pull Census demographics for three towns simultaneously
- Create choropleth maps of population and income across rural block groups
- Discover healthcare, education, and shopping POIs
- Measure the walk-vs-drive equity gap in a rural town
- Run a formal multi-location comparison
- Model a hospital closure vulnerability scenario
- Import custom facility data from CSV
- Generate a shareable HTML report

---

## Why 25 Minutes?

Urban accessibility studies commonly use a **15-minute walk** or drive as the threshold for "convenient" access. In rural areas, that standard is unrealistic. Residents of western Kansas routinely drive 20--30 minutes one way for groceries, medical appointments, or school events. The Kansas Department of Transportation considers a 30-minute drive to essential services an acceptable rural standard.

We use **25 minutes** as a moderate threshold -- long enough to capture realistic rural commuting patterns, short enough to distinguish between towns that serve nearby residents well and those that leave outlying areas underserved.

| Setting | Typical threshold | Mode |
|---|---|---|
| **Urban** | 15 minutes | Walk |
| **Suburban** | 15 minutes | Drive |
| **Rural** | 25 minutes | Drive |

A 25-minute drive on Great Plains highways can cover 20--25 miles depending on the road network. Compare that to a typical Kansas county, which averages about 720 square miles (roughly 27 miles on a side). The isochrone and the county are similar in scale, but they are shaped very differently: the isochrone follows the road network, while the county is a rectangle.

---

## Setup and Imports

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper @ git+https://github.com/mihiarc/socialmapper.git'

In [ ]:
from socialmapper import (
    create_isochrone,
    get_census_blocks,
    get_census_data,
    create_map,
    get_poi,
    analyze_multiple_pois,
    generate_report,
    import_poi_csv,
)

import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display, HTML

# Consistent plot styling for the entire notebook
plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

---

## Step 1: Define the Three Hub Towns

We use central coordinates for each town. These points serve as the **origin** for isochrone generation, POI searches, and census queries.

In [ ]:
towns = {
    "Hays, KS":       (38.8794, -99.3268),
    "Dodge City, KS": (37.7528, -100.0171),
    "Liberal, KS":    (37.0439, -100.9210),
}

context = pd.DataFrame({
    "Town": ["Hays", "Dodge City", "Liberal"],
    "Approx. Pop.": ["~21,000", "~28,000", "~20,000"],
    "Character": [
        "University town (Fort Hays State), regional medical center",
        "Agricultural hub, meatpacking, county seat of Ford County",
        "Meatpacking and energy, near Oklahoma border",
    ],
    "Key Employers": [
        "FHSU, Hays Medical Center",
        "Cargill, National Beef, USD 443",
        "Seaboard Foods, NatGas Midstream",
    ],
})
display(context)

for name, (lat, lon) in towns.items():
    print(f"{name}: lat={lat}, lon={lon}")

---

## Step 2: Generate 25-Minute Driving Isochrones

An **isochrone** is a polygon enclosing all the area reachable from a starting point within a given travel time. Unlike a circular buffer, isochrones account for the actual road network -- they stretch along highways and contract where roads are sparse.

We use **driving mode** with a **25-minute** threshold because:

1. In rural western Kansas, virtually all daily trips are by car. There is no public transit.
2. A 25-minute drive captures the realistic service area of a hub town.
3. The resulting polygon reveals how the highway network shapes access -- US-183, US-283, US-56, and US-83 are the main corridors here.
4. Comparing isochrone shape and area to county boundaries shows how administrative and functional geographies diverge.

SocialMapper uses the **Valhalla** open-source routing engine, which models driving on the actual OpenStreetMap road network.

In [ ]:
isochrones = {}

for name, coords in towns.items():
    iso = create_isochrone(coords, travel_time=25, travel_mode="drive")
    isochrones[name] = iso
    area = iso["properties"]["area_sq_km"]
    print(f"{name}: {area:.1f} sq km reachable within a 25-minute drive")
    time.sleep(1)  # Rate-limit courtesy pause

print()
areas = {n: isochrones[n]["properties"]["area_sq_km"] for n in towns}
largest = max(areas, key=areas.get)
smallest = min(areas, key=areas.get)
print(f"Largest driving area: {largest} ({areas[largest]:.1f} sq km)")
print(f"Smallest driving area: {smallest} ({areas[smallest]:.1f} sq km)")
print(f"Ratio: {areas[largest] / areas[smallest]:.2f}x")
print("\nDifferences in area reflect highway density, road quality, and speed limits.")

---

## Step 3: Isochrone Area vs. Kansas County Area

Kansas has 105 counties. The average Kansas county is about **2,145 sq km** (828 sq mi). Western Kansas counties tend to be larger than eastern ones.

How do our 25-minute driving isochrones compare? If the isochrone is much smaller than a county, that tells us the town's functional service area covers only a fraction of its home county. If it is larger, the town's reach extends into neighboring counties -- and the county boundary is a poor proxy for community membership.

In [ ]:
avg_ks_county_area = 2145  # sq km (828 sq mi)

town_names = list(towns.keys())
iso_areas = [isochrones[n]["properties"]["area_sq_km"] for n in town_names]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(town_names))
width = 0.35

bars_iso = ax.bar(x - width / 2, iso_areas, width, label="25-min Drive Isochrone",
                  color="#4c78a8", edgecolor="white", linewidth=1.2)
bars_county = ax.bar(x + width / 2, [avg_ks_county_area] * len(town_names), width,
                     label="Avg. KS County", color="#e45756", edgecolor="white", linewidth=1.2)

for bar, val in zip(bars_iso, iso_areas):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f"{val:,.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
for bar in bars_county:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f"{avg_ks_county_area:,}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels([n.replace(", KS", "") for n in town_names])
ax.set_ylabel("Area (sq km)")
ax.set_title("25-Minute Driving Area vs. Average Kansas County", fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

for name in town_names:
    ratio = isochrones[name]["properties"]["area_sq_km"] / avg_ks_county_area
    print(f"{name}: isochrone is {ratio:.0%} the size of an average KS county")

### Interpreting the Comparison

If the isochrone is smaller than the county average, the town's 25-minute driving community covers less ground than a single county. If it is similar in size, the isochrone roughly replaces the county as a service boundary. If it is larger, the town draws from multiple counties -- a strong signal that county-level data misrepresents the real community.

In all three cases, the *shape* matters as much as the size. These isochrones are not circles -- they are elongated along highways and pinched where the road network is thin. That is the point: community boundaries follow roads, not survey lines.

---

## Step 4: Fetch Census Demographics

To understand *who lives* in each town's functional community, we pull five American Community Survey (ACS) variables:

| Variable | What it tells us |
|---|---|
| `population` | Total residents in each block group |
| `median_income` | Household purchasing power |
| `poverty` | Count of residents below the federal poverty line |
| `housing_units` | Total housing stock (proxy for density) |
| `households_no_vehicle` | Residents who depend entirely on others for transportation |

The `get_census_data` function fetches data at the **block group** level -- the smallest geography for which the Census Bureau publishes most ACS estimates.

In [ ]:
demographic_variables = [
    "population",
    "median_income",
    "poverty",
    "housing_units",
    "households_no_vehicle",
]

blocks_data = {}
census_data = {}
merged_data = {}

for name in towns:
    iso = isochrones[name]

    # Fetch block group boundaries that intersect the isochrone
    blocks = get_census_blocks(polygon=iso)

    # Fetch ACS demographic data for those block groups
    census = get_census_data(iso, variables=demographic_variables)

    # Merge geometry + demographics into a single list of dicts
    merged = []
    for block in blocks:
        geoid = block["geoid"]
        if geoid in census.data:
            merged.append({**block, **census.data[geoid]})

    blocks_data[name] = blocks
    census_data[name] = census
    merged_data[name] = merged

    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    print(f"\n{name}:")
    print(f"  Block groups matched: {len(merged)}")
    print(f"  Total population:     {total_pop:,}")

---

## Step 5: Demographic Summary Table

Let us compute aggregate statistics for each town and display them in a comparison table.

In [ ]:
summary = {}

for name in towns:
    census = census_data[name]
    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    incomes = [
        d["median_income"]
        for d in census.data.values()
        if d.get("median_income") is not None and d["median_income"] > 0
    ]
    avg_income = sum(incomes) / len(incomes) if incomes else 0
    total_poverty = sum(
        d.get("poverty", 0)
        for d in census.data.values()
        if d.get("poverty") is not None
    )
    poverty_rate = (total_poverty / total_pop * 100) if total_pop > 0 else 0
    no_vehicle = sum(
        d.get("households_no_vehicle", 0)
        for d in census.data.values()
        if d.get("households_no_vehicle") is not None
    )
    total_housing = sum(
        d.get("housing_units", 0)
        for d in census.data.values()
        if d.get("housing_units") is not None
    )

    summary[name] = {
        "Population": total_pop,
        "Avg. Median Income": f"${avg_income:,.0f}",
        "Poverty Count": total_poverty,
        "Poverty Rate": f"{poverty_rate:.1f}%",
        "Households w/o Vehicle": no_vehicle,
        "Housing Units": total_housing,
        "Drive Area (sq km)": f"{isochrones[name]['properties']['area_sq_km']:.1f}",
    }

summary_df = pd.DataFrame(summary)
display(summary_df)

### Reading the Summary

Pay attention to how these three towns differ:

- **Hays** is a university town with Fort Hays State University and a regional medical center. It may show higher median incomes and lower poverty rates, reflecting the stabilizing effect of higher education and healthcare employment.
- **Dodge City** and **Liberal** are both meatpacking towns, drawing large immigrant workforces. They may show lower median incomes and higher poverty rates, but also younger populations and larger household sizes.
- **Households without vehicles** is a critical metric in rural areas. Without a car in western Kansas, you are effectively stranded. Even a few hundred car-free households represent a serious equity concern in a region with no public transit.

---

## Step 6: Population Choropleth Maps

Choropleth maps color each block group by a numeric variable. The dashed boundary shows the 25-minute driving isochrone -- the functional community boundary.

In [ ]:
for name in towns:
    map_result = create_map(
        data=merged_data[name],
        column="population",
        title=f"Population by Block Group -- {name}",
        overlay_boundary=isochrones[name],
        show_stats=True,
    )
    print(f"\n--- {name} ---")
    display(Image(data=map_result.image_data))

### Reading the Population Maps

Each block group is shaded by total population -- darker areas are more densely populated. The **dashed boundary** marks the 25-minute driving isochrone.

In rural areas, you will typically see:
- A few **dark block groups** in town, where most people live
- **Light or empty block groups** in the surrounding countryside, where population is very sparse
- The isochrone extending well beyond the town itself, capturing the commuting shed

Rural block groups are physically enormous (sometimes an entire county) because the Census Bureau needs a minimum population per block group. This means a single block group may cover hundreds of square miles of farmland.

---

## Step 7: Income Choropleth Maps

Income maps reveal economic stratification within each driving area. We use the `RdYlGn` (red-yellow-green) colormap so that low-income areas appear in red and high-income areas in green.

In [ ]:
for name in towns:
    income_map = create_map(
        data=merged_data[name],
        column="median_income",
        title=f"Median Household Income -- {name}",
        overlay_boundary=isochrones[name],
        show_stats=True,
        cmap="RdYlGn",
    )
    print(f"\n--- {name} ---")
    display(Image(data=income_map.image_data))

### Reading the Income Maps

Look for the spatial distribution of income within each town's community:

- **Hays** may show a gradient from higher incomes near the university and medical center to lower incomes on the periphery.
- **Dodge City** and **Liberal** may show generally lower median incomes, reflecting their meatpacking economies. Within each town, some block groups near the plants may have notably lower incomes.
- **Rural block groups** outside the towns themselves can go either way -- farming families may have high asset wealth but variable income depending on crop prices.

Remember: `median_income` is reported per block group. A single block group's median can mask wide variation among individual households.

---

## Step 8: Service Discovery -- Healthcare, Education, Shopping

In rural communities, access to services defines quality of life. We search OpenStreetMap for three critical categories:

- **Healthcare**: hospitals, clinics, pharmacies, dentists
- **Education**: schools, libraries, universities
- **Shopping**: supermarkets, grocery stores, convenience stores, general retail

The `get_poi` function creates a 25-minute driving isochrone behind the scenes, queries the Overpass API for matching OSM features, and computes actual driving travel times via Valhalla's matrix API.

In [ ]:
service_categories = ["healthcare", "education", "shopping"]

poi_data = {}

for name, coords in towns.items():
    pois = get_poi(
        coords,
        categories=service_categories,
        travel_time=25,
        travel_mode="drive",
        limit=80,
    )
    poi_data[name] = pois
    print(f"\n{'=' * 55}")
    print(f"{name}: {len(pois)} service POIs within 25-min drive")
    print(f"{'=' * 55}")

    # Count by category
    cat_counts = {}
    for p in pois:
        cat = p.get("category", "other")
        cat_counts[cat] = cat_counts.get(cat, 0) + 1
    for cat, count in sorted(cat_counts.items()):
        print(f"  {cat:<15} {count}")

    # Show top 5 closest
    print(f"\n  {'Name':<35} {'Category':<15} {'Drive (min)'}")
    print(f"  {'-'*35} {'-'*15} {'-'*10}")
    for p in pois[:5]:
        travel = p.get("travel_time_minutes", "N/A")
        print(f"  {p['name'][:34]:<35} {p['category']:<15} {travel}")
    time.sleep(3)  # Rate-limit courtesy pause between towns

---

## Step 9: Service Comparison Charts and POI Overlay Maps

A grouped bar chart makes differences in service availability immediately visible. We then overlay POI markers on the population maps for spatial context.

In [ ]:
# Grouped bar chart: services by category
categories_for_chart = ["healthcare", "education", "shopping"]
town_names_short = [n.replace(", KS", "") for n in towns]
colors = ["#4c78a8", "#f58518", "#54a24b"]

category_counts = {}
for name in towns:
    counts = {}
    for p in poi_data[name]:
        cat = p.get("category", "other")
        if cat in categories_for_chart:
            counts[cat] = counts.get(cat, 0) + 1
    category_counts[name] = counts

x = np.arange(len(categories_for_chart))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
for i, (name, color) in enumerate(zip(towns, colors)):
    vals = [category_counts[name].get(cat, 0) for cat in categories_for_chart]
    bars = ax.bar(x + i * width, vals, width, label=name.replace(", KS", ""),
                  color=color, edgecolor="white", linewidth=1.2)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    str(val), ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + width)
ax.set_xticklabels([c.replace("_", " ").title() for c in categories_for_chart])
ax.set_ylabel("Number of POIs")
ax.set_title("Services Within 25-Minute Drive by Category", fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# POI overlay maps for each town
for name in towns:
    overlay_points = [
        {"lat": p["lat"], "lon": p["lon"], "name": p["name"]}
        for p in poi_data[name][:20]
    ]

    map_result = create_map(
        data=merged_data[name],
        column="population",
        title=f"Services Overlay -- {name}",
        overlay_boundary=isochrones[name],
        overlay_points=overlay_points,
        show_stats=True,
    )
    print(f"\n--- {name} ---")
    display(Image(data=map_result.image_data))

### Interpreting the Service Maps

Look for:
- **Clustering of services in town** -- most POIs will cluster in the town center, leaving outlying areas in the isochrone with no nearby services.
- **Highway corridor effects** -- some services (gas stations, convenience stores) may appear along highway corridors outside town.
- **Gaps in coverage** -- large areas within the isochrone that have no POI markers represent service deserts. For residents living in those areas, even the 25-minute drive may not bring them to a hospital or grocery store.

---

## Step 10: Walk vs. Drive Equity Gap (Hays)

Hays is a university town with a relatively compact downtown. Let us compare what residents can reach by **walking 15 minutes** versus **driving 25 minutes**. This reveals the equity gap between those who can drive and those who cannot.

In rural towns, this gap is often enormous. A resident without a car in Hays may be able to walk to a convenience store and a pharmacy, but the hospital, Walmart, and high school are all beyond walking range.

In [ ]:
hays_coords = towns["Hays, KS"]

# Walking isochrone (15 min)
walk_iso = create_isochrone(hays_coords, travel_time=15, travel_mode="walk")
walk_area = walk_iso["properties"]["area_sq_km"]

# Driving isochrone (25 min) -- already computed
drive_area = isochrones["Hays, KS"]["properties"]["area_sq_km"]

print(f"Hays, KS -- Walk vs. Drive Comparison")
print(f"  15-min walk area:  {walk_area:.2f} sq km")
print(f"  25-min drive area: {drive_area:.1f} sq km")
print(f"  Drive area is {drive_area / walk_area:.0f}x the walk area")

time.sleep(3)  # Rate-limit courtesy pause

# Walking POIs
walk_pois = get_poi(
    hays_coords,
    categories=service_categories,
    travel_time=15,
    travel_mode="walk",
    limit=50,
)
drive_pois = poi_data["Hays, KS"]

print(f"\n  Services within 15-min walk:  {len(walk_pois)}")
print(f"  Services within 25-min drive: {len(drive_pois)}")

if len(walk_pois) > 0:
    print(f"  Driving provides {len(drive_pois) / len(walk_pois):.1f}x the service access")
else:
    print(f"  No services within walking distance -- complete car dependency")

In [ ]:
# Walk vs. Drive bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Area comparison
bars = axes[0].bar(["Walk (15 min)", "Drive (25 min)"], [walk_area, drive_area],
                   color=["#f58518", "#4c78a8"], edgecolor="white", linewidth=1.2)
for bar, val in zip(bars, [walk_area, drive_area]):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                 f"{val:.1f}", ha="center", va="bottom", fontweight="bold", fontsize=11)
axes[0].set_ylabel("Area (sq km)")
axes[0].set_title("Reachable Area -- Hays, KS", fontweight="bold")
axes[0].spines[["top", "right"]].set_visible(False)

# Service count comparison
bars = axes[1].bar(["Walk (15 min)", "Drive (25 min)"], [len(walk_pois), len(drive_pois)],
                   color=["#f58518", "#4c78a8"], edgecolor="white", linewidth=1.2)
for bar, val in zip(bars, [len(walk_pois), len(drive_pois)]):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 str(val), ha="center", va="bottom", fontweight="bold", fontsize=11)
axes[1].set_ylabel("Number of Services")
axes[1].set_title("Service Access -- Hays, KS", fontweight="bold")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

print("In a town with no public transit, this gap defines who has access and who does not.")

---

## Step 11: Multi-Town Formal Comparison

The `analyze_multiple_pois` function performs a structured, side-by-side analysis of multiple locations in a single call. Here we demonstrate it with a single location (to keep API usage light), then build the three-way comparison from the data we have already collected.

In [ ]:
time.sleep(10)  # Rate-limit courtesy pause

# Demonstrate analyze_multiple_pois with a single location
# (In practice you would pass all locations; we keep it light for the tutorial)
comparison = analyze_multiple_pois(
    locations=[towns["Hays, KS"]],
    travel_time=25,
    travel_mode="drive",
    variables=["population", "median_income", "poverty", "housing_units"],
)

print("analyze_multiple_pois returns a structured dict:")
print(f"  Keys: {list(comparison.keys())}")
print(f"  Locations analyzed: {len(comparison['locations'])}")
print(f"  Metadata: {comparison['metadata']}")

hays_result = comparison["locations"][0]
print(f"\n  Hays, KS aggregated data:")
for var, stats in hays_result["aggregated"].items():
    print(f"    {var}: total={stats['total']:,.0f}, mean={stats['mean']:,.0f}, "
          f"min={stats['min']:,.0f}, max={stats['max']:,.0f}")

# Full three-town comparison from data already in memory
print("\n\n" + "=" * 65)
print("THREE-TOWN COMPARISON (from cached census data)")
print("=" * 65)

compare_vars = ["population", "median_income", "poverty", "housing_units"]
for var in compare_vars:
    print(f"\n--- {var.upper()} ---")
    print(f"  {'Town':<20} {'Total':>12}  {'Mean':>10}")
    rows = []
    for name in towns:
        values = [
            d.get(var, 0)
            for d in census_data[name].data.values()
            if d.get(var) is not None
        ]
        total = sum(values)
        mean = total / len(values) if values else 0
        rows.append((name, total, mean))
        print(f"  {name:<20} {total:>12,.0f}  {mean:>10,.0f}")
    highest = max(rows, key=lambda r: r[1])
    lowest = min(rows, key=lambda r: r[1])
    print(f"  Highest: {highest[0]}  |  Lowest: {lowest[0]}")

---

## Step 12: Hospital Closure Vulnerability Analysis

Rural hospital closures are a growing crisis across the Great Plains. Since 2010, dozens of rural hospitals in Kansas have closed or reduced services. When a hospital closes, residents must drive further for emergency care -- and in a heart attack or car accident, those extra minutes can be fatal.

We model this for **Liberal, KS** -- the most geographically isolated of our three towns, sitting near the Oklahoma border:
1. Find healthcare facilities within the 25-minute driving isochrone
2. Expand to a 45-minute driving isochrone to see what alternatives exist
3. Import a CSV of regional facilities to demonstrate `import_poi_csv`

In [ ]:
# Hospital closure scenario for Liberal, KS -- the most isolated of the three towns
liberal_coords = towns["Liberal, KS"]

time.sleep(5)  # Rate-limit courtesy pause

# Current 25-min healthcare
health_25 = get_poi(
    liberal_coords,
    categories=["healthcare"],
    travel_time=25,
    travel_mode="drive",
    limit=30,
)

time.sleep(3)

# Expanded 45-min healthcare (if local hospital closes)
health_45 = get_poi(
    liberal_coords,
    categories=["healthcare"],
    travel_time=45,
    travel_mode="drive",
    limit=50,
)

additional = len(health_45) - len(health_25)

print("=" * 55)
print("Liberal, KS -- Hospital Closure Scenario")
print("=" * 55)
print(f"  Healthcare facilities within 25-min drive: {len(health_25)}")
print(f"  Healthcare facilities within 45-min drive: {len(health_45)}")
print(f"  Additional facilities at 45 min: +{additional}")
if additional > 0:
    print(f"\n  If the closest facility closes, residents gain {additional}")
    print(f"  alternatives -- but at 20+ extra minutes of driving.")
    print(f"\n  New facilities at extended range:")
    existing_names = {p["name"] for p in health_25}
    for p in health_45:
        if p["name"] not in existing_names:
            travel = p.get("travel_time_minutes", "?")
            print(f"    {p['name'][:40]:<42} {travel} min")
else:
    print(f"\n  WARNING: No additional facilities even at 45 minutes.")
    print(f"  This town is critically vulnerable to hospital closure.")

In [ ]:
# Demonstrate import_poi_csv with a small hypothetical CSV of regional facilities
import tempfile, os

csv_content = """name,latitude,longitude,type
Hays Medical Center,38.8726,-99.3372,hospital
Dodge City Medical Center,37.7506,-100.0214,hospital
Southwest Medical Center (Liberal),37.0486,-100.9198,hospital
Russell County Hospital,38.8953,-98.8598,hospital
Ness County Hospital,38.4529,-99.9065,hospital
Hodgeman County Health Center,38.0886,-99.8949,clinic
Meade District Hospital,37.2858,-100.3406,hospital
"""

csv_path = os.path.join(tempfile.gettempdir(), "ks_hospitals.csv")
with open(csv_path, "w") as f:
    f.write(csv_content)

custom_hospitals = import_poi_csv(
    csv_path,
    name_field="name",
    lat_field="latitude",
    lon_field="longitude",
    type_field="type",
)

print(f"Imported {len(custom_hospitals)} regional facilities from CSV:\n")
for h in custom_hospitals:
    print(f"  {h['name']:<40} ({h['lat']:.4f}, {h['lon']:.4f})  [{h['category']}]")

print(f"\nThe import_poi_csv function is useful for integrating local health department")
print(f"data, custom facility lists, or state licensing records that may be more")
print(f"complete than OpenStreetMap for rural areas.")

### Why This Matters

Rural hospital closures create a cascading effect:
1. **Emergency care**: Golden-hour trauma response deteriorates. A 25-minute drive becomes 45 minutes or more.
2. **Workforce**: Healthcare workers leave for towns with open hospitals, further depleting the local economy.
3. **Recruitment**: It becomes harder to attract physicians, teachers, and other professionals to a town without a hospital.
4. **Property values**: Home values decline when the nearest hospital closes.

The `import_poi_csv` function lets analysts integrate authoritative data sources (state licensing databases, Medicare facility lists, local health department records) that are often more complete than OpenStreetMap for rural areas.

---

## Step 13: Generate Shareable HTML Report

The `generate_report` function transforms the comparison dictionary into a formatted, shareable HTML document. This is useful for distributing findings to county commissioners, health departments, or community organizations that may not use Jupyter notebooks.

In [ ]:
report_html = generate_report(comparison, format="html")
print(f"Generated HTML report: {len(report_html):,} characters")
display(HTML(report_html))

---

## Key Findings

Let us consolidate the metrics into a final summary.

In [ ]:
print("=" * 65)
print("RURAL COMMUNITY MAPPING ASSESSMENT")
print("Western Kansas: Hays, Dodge City, Liberal (25-min drive)")
print("=" * 65)

for name in towns:
    census = census_data[name]
    pois = poi_data[name]
    iso = isochrones[name]

    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    incomes = [
        d["median_income"]
        for d in census.data.values()
        if d.get("median_income") is not None and d["median_income"] > 0
    ]
    avg_income = sum(incomes) / len(incomes) if incomes else 0
    total_poverty = sum(
        d.get("poverty", 0)
        for d in census.data.values()
        if d.get("poverty") is not None
    )
    poverty_rate = (total_poverty / total_pop * 100) if total_pop > 0 else 0
    no_vehicle = sum(
        d.get("households_no_vehicle", 0)
        for d in census.data.values()
        if d.get("households_no_vehicle") is not None
    )

    area = iso["properties"]["area_sq_km"]
    county_ratio = area / avg_ks_county_area

    print(f"\n--- {name} ---")
    print(f"  Drive area:              {area:.1f} sq km ({county_ratio:.0%} of avg KS county)")
    print(f"  Population:              {total_pop:,}")
    print(f"  Avg. median income:      ${avg_income:,.0f}")
    print(f"  Poverty rate:            {poverty_rate:.1f}%")
    print(f"  Households w/o vehicle:  {no_vehicle:,}")
    print(f"  Service POIs (25-min):   {len(pois)}")

### Interpretation

1. **Counties are not communities.** The 25-minute driving isochrones reveal that a hub town's functional community follows highways, not county lines. Residents in adjacent counties who live along a highway corridor are functionally part of the hub town's community, while residents in the same county who live far from a highway may be effectively isolated.

2. **University and medical center towns are different.** Hays, with Fort Hays State University and Hays Medical Center, likely shows higher incomes, lower poverty, and more services than the meatpacking towns of Dodge City and Liberal. Anchor institutions stabilize rural economies.

3. **Car dependency is absolute.** The walk-vs-drive comparison in Step 10 shows that walking access covers a tiny fraction of driving access. In rural Kansas, losing a car means losing access to virtually everything. The `households_no_vehicle` count, even if small, represents severe hardship.

4. **Hospital closure is an existential threat.** The 25-to-45-minute expansion in Step 12 shows how thin the safety net is. If a town's hospital closes, the next-nearest facility may be 45 minutes or more away -- well beyond the golden hour for trauma care.

5. **Service concentration creates fragility.** Most POIs cluster tightly in town centers. The vast majority of the isochrone area (farmland, small unincorporated communities) has zero services. This concentration is efficient when the hub town is healthy, but creates system-wide failure if the town declines.

6. **Three towns, three profiles.** Despite similar populations and geographic settings, Hays, Dodge City, and Liberal have meaningfully different demographic and service profiles. Policy interventions (broadband investment, hospital subsidies, transit pilots) should be tailored to each town's specific needs rather than applied uniformly across "rural Kansas."

---

## Limitations and Next Steps

### Data Limitations

- **OpenStreetMap completeness**: OSM coverage in rural western Kansas is less complete than in metro areas. Some businesses may be missing, and hours of operation are rarely tagged. This can make service availability appear *worse* than it actually is.

- **ACS margins of error**: Census data at the block group level in rural areas carries very large margins of error because block groups may contain only 600--1,000 people. Income and poverty estimates should be treated as rough guides, not precise figures.

- **Isochrone assumptions**: Driving isochrones assume posted speed limits and normal road conditions. In winter (ice storms) or during harvest season (slow farm equipment on highways), actual travel times can be significantly longer.

- **Block group size**: Rural block groups are physically enormous. A single block group may span hundreds of square miles, making the choropleth maps less informative than in urban settings where block groups are small and numerous.

- **Seasonal variation**: Western Kansas has significant seasonal population fluctuation due to agricultural labor cycles. ACS estimates may not capture peak meatpacking workforce periods accurately.

### Suggested Extensions

- **More Kansas towns**: Add Garden City, Great Bend, or Salina to `analyze_multiple_pois` for a broader western Kansas comparison.
- **Broadband overlay**: Cross-reference FCC broadband coverage maps with isochrone areas to assess digital access alongside physical access.
- **School district analysis**: Use education POIs to assess whether school consolidation is placing unreasonable driving burdens on families.
- **Temporal analysis**: Compare ACS estimates across multiple years to track whether these communities are growing or declining.
- **Interactive maps**: Use `create_map(..., export_format='html')` for zoomable, clickable maps that county planners can explore in a browser.

---

## API Cheat Sheet

Quick reference for all SocialMapper functions used in this notebook:

| Function | Purpose | Key Parameters |
|---|---|---|
| `create_isochrone(location, travel_time, travel_mode)` | Travel-time polygon from a point | `travel_mode`: `"walk"`, `"drive"`, `"bike"` |
| `get_census_blocks(polygon=iso)` | Census block groups intersecting an area | Pass an isochrone or GeoJSON dict |
| `get_census_data(location, variables)` | ACS demographic data by block group | Variables: `"population"`, `"median_income"`, `"poverty"`, etc. |
| `create_map(data, column, ...)` | Choropleth map with optional overlays | `overlay_boundary`, `overlay_points`, `show_stats`, `cmap` |
| `get_poi(location, categories, ...)` | OpenStreetMap points of interest | `travel_time` triggers isochrone-bounded search |
| `analyze_multiple_pois(locations, ...)` | Multi-location demographic comparison | Returns rankings for each variable |
| `generate_report(data, format)` | Formatted HTML report from analysis data | `format`: `"html"` |
| `import_poi_csv(path, ...)` | Custom POI data from CSV file | Specify column mappings for lat/lon/name/type |

---

*This notebook was created as part of the SocialMapper tutorial series. For more information, see the [SocialMapper documentation](https://github.com/mihiarc/socialmapper).*